In [0]:
%sql
use iot_catalog_p2.gold;

In [0]:
from pyspark.sql.functions import col, count, when

def run_gold_layer_tests():
    print("🚀 Starting Gold Layer Star Schema Validation...")
    
    # 1. Referential Integrity Test (Fact to Dimensions)
    # Checks if any record in the Fact table is missing a link to its Dimensions
    integrity_check = spark.sql("""
        SELECT 
            (SELECT COUNT(*) FROM gold.fact_iot_events f 
             LEFT JOIN gold.dim_sensor_master s ON f.sensor_sk = s.sensor_sk 
             WHERE s.sensor_sk IS NULL) as orphaned_sensors,
            
            (SELECT COUNT(*) FROM gold.fact_iot_events f 
             LEFT JOIN gold.dim_device_master d ON f.device_sk = d.device_sk 
             WHERE d.device_sk IS NULL) as orphaned_devices,
             
            (SELECT COUNT(*) FROM gold.fact_iot_events f 
             LEFT JOIN gold.dim_location_environment l ON f.location_sk = l.location_sk 
             WHERE l.location_sk IS NULL) as orphaned_locations
    """).collect()[0]

    assert integrity_check['orphaned_sensors'] == 0, "❌ Fail: Orphaned sensor keys found in Fact table!"
    assert integrity_check['orphaned_devices'] == 0, "❌ Fail: Orphaned device keys found in Fact table!"
    print("✅ Pass: Star Schema referential integrity verified (No orphans).")

    # 2. Schema Presence Test
    # Ensures all expected Gold tables exist in the catalog
    expected_tables = [
        "fact_iot_events", 
        "dim_sensor_master", 
        "dim_device_master", 
        "dim_location_environment", 
        "dim_time_events"
    ]
    
    for table in expected_tables:
        table_exists = spark.catalog.tableExists(f"gold.{table}")
        assert table_exists, f"❌ Fail: Table gold.{table} is missing!"
    print(f"✅ Pass: All {len(expected_tables)} Gold tables are present in the catalog.")

    # 3. Data Quality: Metric Bound Test
    # Ensures the analytical values in the Fact table are within logical bounds
    metrics_check = spark.table("gold.fact_iot_events").select(
        count(when(col("cpu_usage_percent") > 100, 1)).alias("invalid_cpu"),
        count(when(col("battery_voltage") < 0, 1)).alias("invalid_battery")
    ).collect()[0]

    assert metrics_check['invalid_cpu'] == 0, "❌ Fail: CPU usage found above 100%!"
    assert metrics_check['invalid_battery'] == 0, "❌ Fail: Negative battery voltage found!"
    print("✅ Pass: Hardware metrics (CPU, Battery) are within valid ranges.")

    # 4. Identity Column Check
    # Verifies that Surrogate Keys (SKs) were generated correctly
    sensor_sk_check = spark.table("gold.dim_sensor_master").filter("sensor_sk IS NULL").count()
    assert sensor_sk_check == 0, "❌ Fail: Null Surrogate Keys found in dim_sensor_master!"
    print("✅ Pass: Identity columns (Surrogate Keys) generated correctly.")

run_gold_layer_tests()